# 第 19 课｜什么是 connectome？

到现在，我们已经有了移动 spike event 与 synapse record 的计算框架。下一步的问题是：一个大型真实网络从哪里来？

今天只问一个问题：

> **connectome 给了我们什么信息，又没有给我们什么信息？**

本课主要新概念：**connectome 是“有向图 + metadata（元数据）”。**

## 1. 概念账本

**已经知道：** neuron ID、稀疏 synapse record、source→target edge、event-driven update。

**今天学习：** **连接组（connectome）**：描述哪些神经元连接到哪些神经元的数据集，通常还带有 annotation / metadata。

**只预告：** MaleCNS 转换、manifest/checksum、binary image layout 与全规模装载。

## 2. connectome 是结构，不是正在运行的大脑

在本课程里，可以把 connectome 表示成：

- **node**：由稳定 neuron ID 标识的神经元；
- **directed edge**：source neuron → target neuron 的有向连接；
- **edge metadata**：例如经过批准转换规则得到的 weight 或 synapse count；
- **node metadata**：例如可用的 cell type、region annotation。

connectome 本身**不会**告诉我们当前 membrane voltage、当前 spike queue、random seed 或 sensory input。这些属于运行中的模型或实验状态。

## 3. 一个小图

<div style="max-width:760px; margin:1rem auto;">
<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 760 300" role="img" aria-label="connectome directed graph" style="width:100%; height:auto; display:block;">
  <defs>
    <marker id="l19-arrow" markerWidth="10" markerHeight="10" refX="9" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#2f5f3f"/>
    </marker>
  </defs>
  <g font-family="sans-serif" font-size="24" text-anchor="middle">
    <rect x="70" y="105" width="180" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="160" y="148" fill="#1f2d24">neuron 101</text>
    <rect x="290" y="35" width="180" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="380" y="78" fill="#1f2d24">neuron 205</text>
    <rect x="510" y="105" width="180" height="70" rx="8" fill="#e6f4ea" stroke="#3f7a50" stroke-width="2"/>
    <text x="600" y="148" fill="#1f2d24">neuron 330</text>
  </g>
  <path d="M250 125 C275 100,285 90,300 82" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l19-arrow)"/>
  <path d="M250 150 L510 150" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l19-arrow)"/>
  <path d="M470 82 C500 92,520 108,535 118" fill="none" stroke="#2f5f3f" stroke-width="3" marker-end="url(#l19-arrow)"/>
</svg>
</div>

箭头方向有意义：source 与 target 不能随意交换。

## 4. Run：总结一个极小 connectome

下面只是教学用小数据，**不是 MaleCNS 数据**。

In [ ]:
neurons = [
    {"id": 101, "type": "sensory-like"},
    {"id": 205, "type": "interneuron-like"},
    {"id": 330, "type": "output-like"},
]

edges = [
    (101, 205),
    (101, 330),
    (205, 330),
]

in_degree = {neuron["id"]: 0 for neuron in neurons}
out_degree = {neuron["id"]: 0 for neuron in neurons}

for source, target in edges:
    out_degree[source] += 1
    in_degree[target] += 1

print("neurons:", len(neurons))
print("directed edges:", len(edges))
print("in-degree:", in_degree)
print("out-degree:", out_degree)

## 5. Observe

在这个小图中，neuron 101 有两条 outgoing edge，没有 incoming edge；neuron 330 有两条 incoming edge，没有 outgoing edge。

这些 degree 描述的是**图结构**，并不说明任何神经元现在是否正在 spike。

## 6. ID、metadata 与 dynamic state 是三类东西

一个很重要的分离是：

- **neuron ID**：标识“是谁”；
- **metadata**：描述这个 neuron 或 connection；
- **model state**：随着 simulation / FPGA run 不断变化。

如果把它们混在一起，就很难保证可重复。converter 在翻译连接数据时，不应该偷偷替你发明 membrane state。

## 7. Try It

加入 edge `(330, 101)`。运行前先预测：哪两个 degree entry 会改变？

然后再回答：仅仅因为图结构改变，任何 neuron 的 membrane state 会自动改变吗？

## 8. 作业

[第 19 课作业：计算 in-degree 与 out-degree](../../exercises/zh/19_what_is_connectome.ipynb)

## 9. AI Task

让 AI 提议一种 Python 数据结构来保存 node、directed edge 和 metadata。然后检查：它有没有把 connectivity metadata 与 dynamic neuron state 混在一起？

## 10. Human Check

解释 neuron ID、directed synapse edge、metadata 与 dynamic model state 的区别。为什么交换 source 与 target 会改变 connectome 的含义？

## 11. Engineering Handoff

对应 `RMD-017` 与 `MOD-011 connectome_converter`。正式 converter 最终要把 MaleCNS source data 转成 versioned、FPGA 可消费的 image，其中包括 neuron metadata、source index、synapse records 与 integrity metadata。

## 12. Project Trace

- Lesson：`LSN-019`
- 映射：`RMD-017`
- 需求路径：`TRACE-C-001`
- 正式模块：`MOD-011 connectome_converter`
- 后续 oracle：`T-014 / T-015`

## 13. Exit Ticket

面对一张小连接表，你能够区分 node、directed edge、metadata 与 dynamic state，并计算简单的 graph degree。